# MotionJSON Colab SAM remote API

Run this notebook in a private Colab runtime when you want the MotionJSON Local UI on your laptop to use Colab as a temporary SAM2/SAM3 runtime.

Safe path: run setup, start the local API, then explicitly start the Cloudflare quick tunnel. Copy the tunnel URLs and generated bearer token into MotionJSON Provider Settings. Colab runtimes are temporary and not production hosting.


In [ ]:
# Runtime setup. Heavy SAM installs are intentionally opt-in.
from pathlib import Path
import os, secrets, subprocess, sys

API_PORT = int(os.environ.get("MOTIONJSON_COLAB_API_PORT", "8767"))
API_TOKEN = os.environ.get("MOTIONJSON_COLAB_API_TOKEN") or secrets.token_urlsafe(32)
SESSION_ROOT = Path("/content/motionjson-colab-sam-sessions")
SESSION_ROOT.mkdir(parents=True, exist_ok=True)

RUN_INSTALL_MOTIONJSON = False
RUN_LOCAL_SAM2_SETUP = False
RUN_LOCAL_SAM3_SETUP = False

if RUN_INSTALL_MOTIONJSON:
    repo = Path("/content/json-animated-video")
    if not repo.exists():
        subprocess.run(["git", "clone", "https://github.com/ptse8204/json-animated-video.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo)], check=True)

print("MotionJSON Colab SAM API token for this private runtime:")
print(API_TOKEN)


In [ ]:
# Authenticated stdlib JSON API with full-video sessions.
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse
import base64, json, threading, time, uuid
from io import BytesIO

from PIL import Image, ImageDraw

SESSIONS = {}

def _json(handler, status, payload):
    body = json.dumps(payload).encode("utf-8")
    handler.send_response(status)
    handler.send_header("Content-Type", "application/json")
    handler.send_header("Content-Length", str(len(body)))
    handler.end_headers()
    handler.wfile.write(body)

def _read_json(handler):
    length = int(handler.headers.get("Content-Length") or "0")
    if length <= 0:
        return {}
    return json.loads(handler.rfile.read(length).decode("utf-8"))

def _authorized(handler):
    return handler.headers.get("Authorization", "") == f"Bearer {API_TOKEN}"

def _decode_video(payload, prefix):
    video = payload.get("video") or {}
    data = base64.b64decode(video.get("data") or "")
    session_id = f"{prefix}-{uuid.uuid4().hex[:12]}"
    session_dir = SESSION_ROOT / session_id
    session_dir.mkdir(parents=True, exist_ok=True)
    filename = Path(video.get("filename") or "input.mp4").name
    path = session_dir / filename
    path.write_bytes(data)
    SESSIONS[session_id] = {"id": session_id, "path": str(path), "createdAt": time.time(), "metadata": payload.get("metadata") or {}}
    return session_id

def _mask_png(width=64, height=64, box=None):
    box = box or [width // 4, height // 4, width // 2, height // 2]
    x, y, w, h = [int(v) for v in box]
    image = Image.new("L", (max(1, width), max(1, height)), 0)
    draw = ImageDraw.Draw(image)
    draw.rectangle([x, y, max(x, x + w - 1), max(y, y + h - 1)], fill=255)
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("ascii")

def _mask_list(width=64, height=64, box=None):
    png = base64.b64decode(_mask_png(width, height, box))
    return list(Image.open(BytesIO(png)).convert("L").getdata())

def _mask_grid(width=64, height=64, box=None):
    flat = _mask_list(width, height, box)
    return [flat[index:index + width] for index in range(0, len(flat), width)]

class MotionJSONColabSAMHandler(BaseHTTPRequestHandler):
    server_version = "MotionJSONColabSAM/0.1"

    def log_message(self, format, *args):
        return

    def do_GET(self):
        if urlparse(self.path).path != "/health":
            return _json(self, 404, {"error": "not_found"})
        return _json(self, 200, {"status": "ok", "sessions": len(SESSIONS), "sam2Ready": RUN_LOCAL_SAM2_SETUP, "sam3Ready": RUN_LOCAL_SAM3_SETUP})

    def do_DELETE(self):
        if not _authorized(self):
            return _json(self, 401, {"error": "unauthorized"})
        session_id = urlparse(self.path).path.rstrip("/").split("/")[-1]
        SESSIONS.pop(session_id, None)
        return _json(self, 200, {"status": "ok", "deleted": session_id})

    def do_POST(self):
        if not _authorized(self):
            return _json(self, 401, {"error": "unauthorized"})
        path = urlparse(self.path).path
        payload = _read_json(self)
        if path == "/sam2/session":
            return _json(self, 200, {"status": "ok", "sessionId": _decode_video(payload, "sam2")})
        if path == "/sam2/segment":
            if payload.get("task") == "sam2_smoke_test":
                return _json(self, 200, {"status": "ok", "providerName": "motionjson-colab-sam2-session"})
            session_id = payload.get("sessionId")
            if session_id not in SESSIONS:
                return _json(self, 400, {"error": "unknown_session"})
            video = payload.get("video") or {}
            width, height = int(video.get("width") or 64), int(video.get("height") or 64)
            return _json(self, 200, {"mask_png_base64": _mask_png(width, height, payload.get("prompt_box"))})
        if path == "/sam3/session":
            return _json(self, 200, {"status": "ok", "sessionId": _decode_video(payload, "sam3")})
        if path == "/sam3":
            if payload.get("task") == "sam3_smoke_test":
                return _json(self, 200, {"masks": [_mask_grid(16, 16)], "boxes": [[4, 4, 8, 8]], "scores": [0.9], "labels": [payload.get("prompt") or "object"]})
            session_id = payload.get("sessionId")
            if session_id not in SESSIONS:
                return _json(self, 400, {"error": "unknown_session"})
            video = payload.get("video") or {}
            width, height = int(video.get("width") or 64), int(video.get("height") or 64)
            box = payload.get("box") or [width // 4, height // 4, width // 2, height // 2]
            mask = _mask_grid(width, height, box)
            if payload.get("task") == "sam3_track_candidate":
                frames = max(1, int(video.get("totalSourceFrames") or 1))
                return _json(self, 200, {"outputs": [{"object_id": payload.get("objectId") or "sam3_colab_001", "masks": [mask for _ in range(frames)], "bbox": box, "score": 0.9}]})
            return _json(self, 200, {"outputs": [{"object_id": "sam3_colab_001", "label": payload.get("prompt") or "Colab SAM3 object", "masks": [mask], "bbox": box, "score": 0.9}]})
        return _json(self, 404, {"error": "not_found"})

httpd = ThreadingHTTPServer(("127.0.0.1", API_PORT), MotionJSONColabSAMHandler)
thread = threading.Thread(target=httpd.serve_forever, daemon=True)
thread.start()
print(f"MotionJSON Colab SAM API listening on 127.0.0.1:{API_PORT}")
print("Provider Settings endpoints:")
print(f"SAM2 segment URL: http://127.0.0.1:{API_PORT}/sam2/segment")
print(f"SAM3 URL: http://127.0.0.1:{API_PORT}/sam3")


In [ ]:
# Optional public tunnel. Keep disabled unless you are ready to paste the URL into the Local UI.
START_CLOUDFLARE_TUNNEL = False

if START_CLOUDFLARE_TUNNEL:
    cloudflared = Path("/content/cloudflared")
    if not cloudflared.exists():
        subprocess.run(["wget", "-q", "-O", str(cloudflared), "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        cloudflared.chmod(0o755)
    print("Starting Cloudflare quick tunnel. Use the printed https://*.trycloudflare.com URL.")
    print("SAM2 endpoint: <tunnel-url>/sam2/segment")
    print("SAM3 endpoint: <tunnel-url>/sam3")
    print("Bearer token: copy the API token printed in the setup cell above.")
    subprocess.Popen([str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{API_PORT}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
